Clonar o repositorio

In [22]:
!git clone https://github.com/carbon-footprint-analysis/carbon-footprint-analysis.git

fatal: destination path 'carbon-footprint-analysis' already exists and is not an empty directory.


Instalar a versão especifica para o projeto (VsCode x Colab)

In [23]:
pip install scikit-learn==1.8.0

Importar as bibliotecas

In [24]:
import joblib
import urllib.request
import pandas as pd

Importando o arquivo joblib

In [25]:
import os

# Ver onde você está agora
print("Diretório atual:", os.getcwd())

# Procurar o arquivo .joblib em todo o projeto
for root, dirs, files in os.walk('.'):
    for file in files:
        if file.endswith('.joblib'):
            print("Encontrado:", os.path.join(root, file))

Diretório atual: /content
Encontrado: ./carbon-footprint-analysis/models/carbon_footprint_rf_v1.joblib
Encontrado: ./carbon-footprint-analysis/models/best_carbon_footprint_model.joblib


In [31]:
import os

# Detecta automaticamente o caminho correto do modelo
# Funciona tanto localmente (VS Code / Jupyter) quanto no Google Colab

def find_model_path():
    candidates = [
        # Caminho relativo (execucao local a partir de notebooks/)
        os.path.join('..', 'models', 'best_carbon_footprint_model.joblib'),
        # Caminho relativo (execucao a partir da raiz do projeto)
        os.path.join('models', 'best_carbon_footprint_model.joblib'),
        # Caminho absoluto Google Colab
        '/content/carbon-footprint-analysis/models/best_carbon_footprint_model.joblib',
        # Caminho absoluto baseado no diretorio atual
        os.path.join(os.getcwd(), '..', 'models', 'best_carbon_footprint_model.joblib'),
    ]
    for path in candidates:
        if os.path.exists(path):
            return os.path.abspath(path)
    return None

model_path = find_model_path()

if model_path:
    model = joblib.load(model_path)
    print(f'Modelo carregado com sucesso!')
    print(f'Caminho: {model_path}')
else:
    print('ERRO: Modelo nao encontrado.')
    print('Certifique-se de que o notebook 04 foi executado e o arquivo')
    print('best_carbon_footprint_model.joblib esta na pasta models/')
    print()
    print('Diretorio atual:', os.getcwd())
    print('Conteudo do diretorio pai:')
    parent = os.path.join(os.getcwd(), '..')
    for item in os.listdir(parent):
        print(' ', item)


Analisando o modelo para sanity

In [32]:
print(model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['consumo_kwh', 'mes']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['estado', 'setor',
                                                   'fonte_energia',
                                                   'season'])])),
                ('regressor',
                 GradientBoostingRegressor(max_depth=5, random_state=42))])


In [33]:
encoder = model.named_steps['preprocessor'].named_transformers_['cat']

encoder.categories_

[array(['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG',
        'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR',
        'RS', 'SC', 'SE', 'SP', 'TO'], dtype=object),
 array(['comercial', 'industrial', 'outros', 'residencial', 'rural'],
       dtype=object),
 array(['eólica', 'hidrelétrica', 'nuclear', 'solar', 'térmica'],
       dtype=object),
 array(['Inverno', 'Outono', 'Primavera', 'Verao'], dtype=object)]

Testando o modelo

In [34]:
df = pd.DataFrame({
    "consumo_kwh": [1200],
    "mes": [6],
    "estado": ["SP"],
    "setor": ["industrial"],
    "fonte_energia": ["hydro"],
    "season": ["Inverno"]
})

model.predict(df)

array([14.11453139])

Fazendo o Wrapper


Utilizando o modelo para fazer o comparativo entre todas as fontes de energia eletrica

In [35]:
def predict_all_sources(model, energy_kwh, month, state, usage_type, season):
    sources = ['hidrelétrica', 'nuclear', 'solar', 'térmica', 'eólica']

    results = {}

    for source in sources:

        df = pd.DataFrame({
            "consumo_kwh": [energy_kwh],
            "mes": [month],
            "estado": [state],
            "setor": [usage_type],
            "fonte_energia": [source],
            "season": [season]
        })

        co2 = model.predict(df)[0]

        results[source] = round(float(co2), 2)

    results_sorted = dict(sorted(results.items(), key=lambda x: x[1]))

    return results_sorted

In [36]:
predict_all_sources(
    model,
    energy_kwh=1200,
    month=6,
    state="SP",
    usage_type="industrial",
    season="Inverno"
)

{'hidrelétrica': 6.02,
 'eólica': 13.8,
 'nuclear': 14.51,
 'solar': 54.2,
 'térmica': 969.39}

Criando comparativo para combustiveis liquidos foi utilizado a media de motores motores modernos. E os valores foram retirados do `PCC Guidelines for National Greenhouse Gas Inventories`, `IEA – International Energy Agency` e `US EPA emission factors`

In [37]:
def liquid_fuel_emissions(energy_kwh):

    fuels = {
        "ethanol": {
            "efficiency": 0.27,
            "emission_factor": 0.20
        },
        "gasoline": {
            "efficiency": 0.30,
            "emission_factor": 0.64
        },
        "diesel": {
            "efficiency": 0.38,
            "emission_factor": 0.73
        }
    }

    results = {}

    for fuel, data in fuels.items():

        efficiency = data["efficiency"]
        factor = data["emission_factor"]

        energy_input = energy_kwh / efficiency
        co2 = energy_input * factor

        results[fuel] = round(co2, 2)

    results_sorted = dict(sorted(results.items(), key=lambda x: x[1]))

    return results_sorted

In [38]:
liquid_fuel_emissions(1200)

{'ethanol': 888.89, 'diesel': 2305.26, 'gasoline': 2560.0}

Unificando fontes geradoras e ordenando do menos poluente para o mais poluente adicionado a % de emissão de Co2 que foi aumentado ou diminuido se comparado com a fonte geradora base do brasil `Hydro`

In [39]:
def compare_energy_sources(model, energy_kwh, month, state, usage_type, season):

    electricity = predict_all_sources(
        model,
        energy_kwh,
        month,
        state,
        usage_type,
        season
    )

    fuels = liquid_fuel_emissions(energy_kwh)

    combined = {**electricity, **fuels}

    ranking = dict(sorted(combined.items(), key=lambda x: x[1]))

    base = ranking["hidrelétrica"]

    result = {}

    for source, value in ranking.items():

        percent = ((value - base) / base) * 100

        result[source] = {
            "co2": round(value, 2),
            "vs_hydro_%": round(percent, 2)
        }

    return result

In [40]:
compare_energy_sources(
    model,
    energy_kwh=1200,
    month=6,
    state="SP",
    usage_type="industrial",
    season="Inverno"
)

{'hidrelétrica': {'co2': 6.02, 'vs_hydro_%': 0.0},
 'eólica': {'co2': 13.8, 'vs_hydro_%': 129.24},
 'nuclear': {'co2': 14.51, 'vs_hydro_%': 141.03},
 'solar': {'co2': 54.2, 'vs_hydro_%': 800.33},
 'ethanol': {'co2': 888.89, 'vs_hydro_%': 14665.61},
 'térmica': {'co2': 969.39, 'vs_hydro_%': 16002.82},
 'diesel': {'co2': 2305.26, 'vs_hydro_%': 38193.36},
 'gasoline': {'co2': 2560.0, 'vs_hydro_%': 42424.92}}

CheckPonit Wrapper

In [41]:
!pip install scikit-learn==1.8.0

!git clone https://github.com/carbon-footprint-analysis/carbon-footprint-analysis.git

import sys
import os

# Funciona tanto no Colab quanto localmente
colab_path = "/content/carbon-footprint-analysis"
local_path = os.path.join(os.path.dirname(os.getcwd()), '')

if os.path.exists(colab_path):
    sys.path.append(colab_path)
else:
    sys.path.append(local_path)

fatal: destination path 'carbon-footprint-analysis' already exists and is not an empty directory.


Criando .py do wrapper

Versão Final wrapper

importando o wrapper diretamente do .py para possivel implementação no FastAPI

In [42]:
!pip install scikit-learn==1.8.0

!git clone https://github.com/carbon-footprint-analysis/carbon-footprint-analysis.git

import sys
import os

# Funciona tanto no Colab quanto localmente
colab_path = "/content/carbon-footprint-analysis"
local_path = os.path.join(os.path.dirname(os.getcwd()), '')

if os.path.exists(colab_path):
    sys.path.append(colab_path)
else:
    sys.path.append(local_path)

fatal: destination path 'carbon-footprint-analysis' already exists and is not an empty directory.


In [43]:
wrapper_code = '''import joblib
import pandas as pd
import os

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
model = joblib.load(os.path.join(BASE_DIR, "models", "best_carbon_footprint_model.joblib"))


def predict_all_sources(model, energy_kwh, month, state, usage_type, season):
    sources = ['hidrelétrica', 'nuclear', 'solar', 'térmica', 'eólica']
    results = {}

    for source in sources:
        df = pd.DataFrame({
            "consumo_kwh": [energy_kwh],
            "mes": [month],
            "estado": [state],
            "setor": [usage_type],
            "fonte_energia": [source],
            "season": [season]
        })
        co2 = model.predict(df)[0]
        results[source] = round(float(co2), 2)

    return dict(sorted(results.items(), key=lambda x: x[1]))


def liquid_fuel_emissions(energy_kwh):
    fuels = {
        "ethanol":  {"efficiency": 0.27, "emission_factor": 0.20},
        "gasoline": {"efficiency": 0.30, "emission_factor": 0.64},
        "diesel":   {"efficiency": 0.38, "emission_factor": 0.73}
    }
    results = {}
    for fuel, data in fuels.items():
        energy_input = energy_kwh / data["efficiency"]
        co2 = energy_input * data["emission_factor"]
        results[fuel] = round(co2, 2)

    return dict(sorted(results.items(), key=lambda x: x[1]))


def compare_energy_sources(energy_kwh, month, state, usage_type, season):
    electricity = predict_all_sources(model, energy_kwh, month, state, usage_type, season)
    fuels = liquid_fuel_emissions(energy_kwh)
    combined = {**electricity, **fuels}
    ranking = dict(sorted(combined.items(), key=lambda x: x[1]))
    base = ranking["hidrelétrica"]
    result = {}
    for source, value in ranking.items():
        percent = ((value - base) / base) * 100
        result[source] = {"co2": round(value, 2), "vs_hydro_%": round(percent, 2)}
    return result
'''

with open("wrapper.py", "w", encoding="utf-8") as f:
    f.write(wrapper_code)

print("wrapper.py salvo!")

wrapper.py salvo!


In [52]:
# Importar o wrapper (funciona local e no Colab)
import sys
import os

# Adiciona a raiz do projeto ao path para importar wrapper.py
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from wrapper import compare_energy_sources
print('wrapper.py importado com sucesso!')


/content/carbon-footprint-analysis


Diversos testes para verificar confiabilidade do wrapper

In [54]:
print(compare_energy_sources(1200, 6,  "SP", "industrial",  "Inverno"))
print(compare_energy_sources(800,  3,  "RJ", "comercial",   "Outono"))     # commercial → comercial
print(compare_energy_sources(1500, 12, "MG", "residencial", "Verao"))      # Verão → Verao
print(compare_energy_sources(600,  9,  "RS", "industrial",  "Primavera"))
print(compare_energy_sources(2000, 1,  "BA", "comercial",   "Verao"))      # commercial → comercial, Verão → Verao
print(compare_energy_sources(400,  7,  "SC", "residencial", "Inverno"))
print(compare_energy_sources(1000, 5,  "GO", "rural",       "Outono"))
print(compare_energy_sources(1800, 10, "PA", "industrial",  "Primavera"))
print(compare_energy_sources(500,  4,  "CE", "comercial",   "Outono"))     # commercial → comercial
print(compare_energy_sources(2200, 8,  "PR", "industrial",  "Inverno"))

{'hidrelétrica': {'co2': 6.02, 'vs_hydro_%': 0.0}, 'eólica': {'co2': 13.8, 'vs_hydro_%': 129.24}, 'nuclear': {'co2': 14.51, 'vs_hydro_%': 141.03}, 'solar': {'co2': 54.2, 'vs_hydro_%': 800.33}, 'ethanol': {'co2': 888.89, 'vs_hydro_%': 14665.61}, 'térmica': {'co2': 969.39, 'vs_hydro_%': 16002.82}, 'diesel': {'co2': 2305.26, 'vs_hydro_%': 38193.36}, 'gasoline': {'co2': 2560.0, 'vs_hydro_%': 42424.92}}
{'hidrelétrica': {'co2': 3.81, 'vs_hydro_%': 0.0}, 'eólica': {'co2': 10.44, 'vs_hydro_%': 174.02}, 'nuclear': {'co2': 11.15, 'vs_hydro_%': 192.65}, 'solar': {'co2': 35.65, 'vs_hydro_%': 835.7}, 'ethanol': {'co2': 592.59, 'vs_hydro_%': 15453.54}, 'térmica': {'co2': 624.65, 'vs_hydro_%': 16295.01}, 'diesel': {'co2': 1536.84, 'vs_hydro_%': 40237.01}, 'gasoline': {'co2': 1706.67, 'vs_hydro_%': 44694.49}}
{'hidrelétrica': {'co2': 9.37, 'vs_hydro_%': 0.0}, 'eólica': {'co2': 18.79, 'vs_hydro_%': 100.53}, 'nuclear': {'co2': 19.5, 'vs_hydro_%': 108.11}, 'solar': {'co2': 65.72, 'vs_hydro_%': 601.39}, 

In [55]:
# Ver a importância de cada feature no modelo
import pandas as pd
import matplotlib.pyplot as plt

feature_names = (
    list(model.named_steps['preprocessor']
         .named_transformers_['num']
         .get_feature_names_out(['consumo_kwh', 'mes']))
    +
    list(model.named_steps['preprocessor']
         .named_transformers_['cat']
         .get_feature_names_out(['estado', 'setor', 'fonte_energia', 'season']))
)

importances = model.named_steps['regressor'].feature_importances_

fi = pd.DataFrame({'feature': feature_names, 'importance': importances})
fi = fi.sort_values('importance', ascending=False)

print(fi.to_string(index=False))

                   feature   importance
               consumo_kwh 7.539813e-01
     fonte_energia_térmica 2.448556e-01
       fonte_energia_solar 7.820476e-04
                       mes 1.204320e-04
fonte_energia_hidrelétrica 4.233222e-05
             season_Outono 3.909668e-05
                 estado_RS 2.898782e-05
                 estado_MG 1.838828e-05
                 estado_MA 1.685500e-05
                 estado_DF 1.453575e-05
                 estado_BA 1.119128e-05
                 estado_AM 8.577925e-06
              season_Verao 8.210548e-06
                 estado_SC 7.496551e-06
                 estado_PA 7.447327e-06
                 estado_RN 7.222637e-06
                 estado_PR 6.291940e-06
          season_Primavera 5.824765e-06
                 estado_ES 5.038464e-06
            season_Inverno 3.865590e-06
                 estado_SP 3.699561e-06
                 estado_AL 3.480973e-06
                 estado_MT 3.366649e-06
                 estado_PB 2.883780e-06


In [57]:
import os

for root, dirs, files in os.walk("/content/carbon-footprint-analysis"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/content/carbon-footprint-analysis/notebooks/consumo_energia_emissoes_br.csv
/content/carbon-footprint-analysis/notebooks/v2_company_size_distribution_by_usage.csv
/content/carbon-footprint-analysis/notebooks/media_consumo_indutria_2025.csv
/content/carbon-footprint-analysis/notebooks/shap_feature_importance.csv
/content/carbon-footprint-analysis/notebooks/v2_seasonality_state_class_month.csv
/content/carbon-footprint-analysis/notebooks/df_gerado_teste.csv
/content/carbon-footprint-analysis/notebooks/consumo_medio_categoria_2025.csv
/content/carbon-footprint-analysis/notebooks/v2_consumption_profiles.csv
/content/carbon-footprint-analysis/data/processed/v2_state_distribution.csv
/content/carbon-footprint-analysis/data/processed/v2_fuel_distribution.csv
/content/carbon-footprint-analysis/data/processed/v2_company_size_distribution_by_usage.csv
/content/carbon-footprint-analysis/data/processed/media_consumo_indutria_2025.csv
/content/carbon-footprint-analysis/data/processed/dados_energia

In [58]:
import pandas as pd

#df = pd.read_csv('../data/processed/synthetic_energy_emissions_dataset.csv')
df = pd.read_csv("data/processed/synthetic_energy_emissions_dataset.csv")

print(df['fonte_energia'].value_counts())
print()
print(df.groupby('fonte_energia')['emissao_co2'].mean().sort_values())

fonte_energia
hidrelétrica    51529
térmica         22801
eólica          15824
solar            8957
nuclear           889
Name: count, dtype: int64

fonte_energia
hidrelétrica       72.274822
eólica            200.980933
nuclear           207.235159
solar             809.334373
térmica         14651.135404
Name: emissao_co2, dtype: float64


In [59]:
# Ver o CO₂ médio normalizado por consumo
df['co2_por_kwh'] = df['emissao_co2'] / df['consumo_kwh']
print(df.groupby('fonte_energia')['co2_por_kwh'].mean().sort_values())

fonte_energia
hidrelétrica    0.004001
eólica          0.011005
nuclear         0.011973
solar           0.045017
térmica         0.820606
Name: co2_por_kwh, dtype: float64


In [61]:
import pandas as pd
#emission_df = pd.read_csv('../data/processed/v2_energy_source_emission_factors.csv')
emission_df = pd.read_csv('/content/carbon-footprint-analysis/data/processed/v2_energy_source_emission_factors.csv')
print(emission_df)

  energy_source  emission_factor
0  hidrelétrica            0.004
1        eólica            0.011
2       nuclear            0.012
3         solar            0.045
4       térmica            0.820
